# Benchmark 3 of 3: compare the results

This notebook reads your runs from `benchmark/runs/` and compares the models on:

- accuracy, using the benchmark's own score;
- how often the answer was exactly right;
- how walks ended;
- cost.

It finishes with one question where two models disagree, step by step.

In [ ]:
BENCHMARK = "webqsp"
RUNS = None   # None compares every run; or a list such as ["run-001", "run-002"]

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

import pandas as pd

from utils import display, load_decisions, load_results

results = load_results(BENCHMARK, RUNS)
print(f"{len(results)} walks from {results['run'].nunique()} run(s) and {results['model'].nunique()} model(s)")

## The results

- **score**: the benchmark's score from 0 to 1, with its 95% interval.
- **exactly correct**: the share of questions whose answer was fully right.
- **hit step limit**: walks that never chose STOP.
- **failed**: walks lost to an API error. These are left out of the score.

In [ ]:
table = display.summary(results)
table

## Accuracy against cost

Up and to the left is better. The cost axis is logarithmic.

In [ ]:
display.plot_score_cost(table)

## On the same questions only

Runs can cover different questions. This table keeps only the questions that every model answered, so the comparison
is like for like.

In [ ]:
answered = results[results["score"].notna()]
common = set.intersection(*(set(g["qid"]) for _, g in answered.groupby(["run", "model"])))
print(f"{len(common)} questions answered by every model")
display.summary(results[results["qid"].isin(common)])

## How the walks ended

In [ ]:
(results.groupby(["run", "model"])["stopped_by"].value_counts(normalize=True)
 .unstack(fill_value=0).map(lambda x: f"{x:.0%}"))

## One question, two models

A question where one model was exactly right and another was not, with both decision logs. `probability` is how sure a
decision model was of its choice; LLMs do not return one.

In [ ]:
pivot = results.pivot_table(index="qid", columns=["run", "model"], values="exact", aggfunc="first")
disagree = pivot[(pivot == True).any(axis=1) & (pivot == False).any(axis=1)]  # noqa: E712
if disagree.empty:
    print("Every model agreed on every question. Try more questions or another model.")
else:
    qid = disagree.index[0]
    print("Question:", results.loc[results["qid"] == qid, "question"].iloc[0])
    for (run, model), exact in disagree.loc[qid].items():
        if pd.isna(exact):
            continue
        print(f"\n{model} ({run}): {'exactly right' if exact else 'wrong'}")
        print(display.walk_log(load_decisions(run, BENCHMARK, model)[qid]).to_string())